# Loka Research Agent - LangGraph Version

Build a research agent about **Loka** step by step with
**[LangGraph](https://langchain-ai.github.io/langgraph/)**.

This notebook is a **guided hands-on exercise** that you will follow throughout the workshop. The agent will be built progressively in three exercises, each adding more capabilities. You will find the following markers in the notebook to guide you:

| Marker               | Meaning                                              |
|----------------------|------------------------------------------------------|
| 🛠️ **Setup**        | Run this once to set up the environment and imports. |
| ✅ **Given**          | Code or instructions provided for you                |
| ✏️ **Your turn**     | Code you need to write to complete the exercise.     |
| 🚀 **Going further** | Optional stretch ideas if you finish early.          |
| 📚 **Hints**         | Links into the LangGraph docs for more information.  |

LangGraph is more explicit than Strands: you write the graph that wires the pieces together instead of handing a model a list of tools. That means more code overall -- so in this notebook **more of it is already given** (state, node functions, the run loop). Your turns are the handful of lines that actually make the difference: adding a tool, binding it to the model, wiring an edge, or turning on memory. The graph shape barely changes between exercises, so once you've wired it in Exercise 1, it's given as already-done from Exercise 2 onward.

The exercises contain detailed instructions on what to do, but you are encouraged to explore and experiment. Refer to the LangGraph documentation for more information on the concepts and APIs used in this notebook.

## 🛠️ Setup

Run this cell once to set up the environment and imports. Make sure you have already configured an `.env` file with your **Anthropic API key** and run `uv sync` to have the project dependencies installed. If you haven't done this yet, follow the instructions in the `README.md` file.

In [3]:
import os, sys
from pathlib import Path

# Find the repo root
ROOT = next(b for b in (Path.cwd(), *Path.cwd().parents) if (b / "shared").is_dir())
sys.path.insert(0, str(ROOT / "shared"))

from dotenv import load_dotenv
load_dotenv(ROOT / ".env")
assert os.getenv("ANTHROPIC_API_KEY"), "Add ANTHROPIC_API_KEY to your .env (copy .env.example)."

from langchain_anthropic import ChatAnthropic
from knowledge_base import search_documents, list_topics
from website import search_website

model = ChatAnthropic(
    model=os.getenv("ANTHROPIC_MODEL", "claude-haiku-4-5-20251001"),
    api_key=os.environ["ANTHROPIC_API_KEY"],
    max_tokens=1024,
    temperature=0.3,
)

print("Model ready:", model.model)

Model ready: claude-haiku-4-5-20251001


## ✅ The knowledge base (given)

Your agent's knowledge lives in `shared/`, already written for you:
`search_documents(query)`, `list_topics()`, and `search_website(query)`. In the
exercises you'll wrap these as **tools**. Run this to see what they return (no API
key needed):

In [2]:
print(list_topics())
print("\n--- search_documents('learning') ---\n")
print(search_documents("learning"))

The Loka knowledge base covers these topics:
- AWS Innovation Partner of the Year
- Time Off and the 5/4 Friday Schedule
- Fully Remote, Work From Anywhere
- In-Person Connection Despite Being Remote
- Cutting-Edge Client Projects
- Multicultural, Global Team
- Learning and Development
- Culture of Innovation and Internal Initiatives

--- search_documents('learning') ---

## Learning and Development
One of Loka's headline goals for 2026 is to become one of the world's best learning organizations. That means real budget and real time for growth: courses, certifications, and the expectation that you keep leveling up. Learning is treated as part of the job, not a side quest.


In [3]:
print("--- search_website('services') ---\n")
print(search_website("what customers does Loka work with?"))

--- search_website('services') ---

(source: live)

[https://www.loka.com/about]
AWS, Jeff oversees Global Direct Sales at Loka, helping SMB customers unlock value through agentic AI, modernization and cloud migration. A culture of innovation Our team members live by the credo What you develop matters. We work wherever we’re most productive and take every other

[https://www.loka.com/]
17 What Fascinates Sol Rashidi? Navigating AI, balancing the possible with the practical and the art of going rogue. Jump to episode Jump to episode Blog Life at Loka • 7.8.26 Loka's Explorer Program: The Best Decision I Almost Didn't Make What an ML Engineer

[https://www.loka.com/]
Cameo. Their deep GenAI, ML and AWS expertise, paired with industry insight and exceptional customer care, makes them a standout among consultancies." Dom Scandinaro CTO , Cameo “In my 35 years of work experience, I have not come across a consulting partner like Loka. They


## Exercise 1: Basic Agent

An agent in LangGraph is a **graph**: a small state machine you wire together
yourself. The state holds the conversation (`messages`), one **node** calls the
model, another **node** runs tools, and a **conditional edge** decides whether to
loop back to the model or stop.

That wiring is the piece Strands' `Agent` class did for you automatically. You'll
build it once, here, in this exercise -- its shape barely changes for the rest of
the notebook, so from Exercise 2 onward it'll already be wired for you.

Your job: turn the given `search_documents` into a tool (same idea as the Strands
version), bind it to the model, and connect the graph: a node that calls the
model, a node that runs the tool, and the router between them.

> 💡 The tool's **docstring** is what the model reads to decide when to use it --
> write it for the model, same as in the Strands notebook.

**📚 Hints**
- [`tools_condition` prebuilt router](https://langchain-ai.github.io/langgraph/reference/prebuilt/#langgraph.prebuilt.tool_node.tools_condition)
- [`ToolNode` prebuilt](https://langchain-ai.github.io/langgraph/reference/prebuilt/#langgraph.prebuilt.tool_node.ToolNode)

### ✅ Given: the state and the system prompt

The **state** is what flows through every node in the graph -- here, just the
running list of messages. `add_messages` tells LangGraph to *append* new
messages to it instead of you managing the list by hand (this is the "State =
Memory" building block the rest of the notebook builds on).

In [4]:
from typing import Annotated, TypedDict

from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage
from langchain_core.tools import tool
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition


class State(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]


SYSTEM_PROMPT = """You are the Loka Research Agent, a friendly assistant that \
answers questions about Loka (the company).

- Always answer from the knowledge base via search_knowledge_base; do not rely \
on prior knowledge about Loka.
- If the knowledge base has no answer, say so instead of guessing.
- Be concise, warm, and a little proud of how great Loka is to work at.
"""

### ✏️ Your turn -- TODO 1: make a tool

Wrap the given `search_documents(query)` as a LangChain `@tool` and add a
docstring for the model to read (same idea as the Strands version -- the
docstring is the model-facing contract).

In [5]:
@tool
def search_knowledge_base(query: str) -> str:
    """Search Loka's internal knowledge base for information about the company,
    its benefits, culture, and how it works.

    Args:
        query: A short natural-language description of what to look for.
    """
    return search_documents(query)

### ✏️ Your turn -- TODO 2: bind the tool to the model

`.bind_tools([...])` is what lets the model *request* a tool call in its
output -- the `ToolNode` you'll wire next is what actually runs it.

In [6]:
model_with_tools = model.bind_tools([search_knowledge_base])

### ✅ Given: the node that calls the model

This node just hands the running state to the model and appends its reply.
It's short and won't change for the rest of the notebook.

In [7]:
def agent_node(state: State) -> dict:
    return {"messages": [model_with_tools.invoke(state["messages"])]}

### ✏️ Your turn -- TODO 3: wire the graph

Two nodes are registered for you below (`agent_node`, and a `ToolNode` running
your tool). Connect them:
- an edge from `START` into `"agent"`
- a **conditional edge** out of `"agent"`, using `tools_condition` as the
  router -- it reads the model's last message and returns `"tools"` if it asked
  for one, otherwise it should end the run
- an edge from `"tools"` back to `"agent"`, so the loop continues after the
  tool runs

In [8]:
builder = StateGraph(State)
builder.add_node("agent", agent_node)
builder.add_node("tools", ToolNode([search_knowledge_base]))

builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", tools_condition, {"tools": "tools", END: END})
builder.add_edge("tools", "agent")

graph = builder.compile()

### ✅ Given: run it

In [9]:
result = graph.invoke({
    "messages": [SystemMessage(SYSTEM_PROMPT), HumanMessage("What is Loka's time-off policy?")]
})

for message in result["messages"]:
    if getattr(message, "tool_calls", None):
        for call in message.tool_calls:
            print(f"[tool call] {call['name']}({call['args']})")
    elif message.type == "tool":
        print(f"[tool result] {message.content}\n")
print(result["messages"][-1].content)

[tool call] search_knowledge_base({'query': 'time-off policy vacation days PTO'})
[tool result] ## Time Off and the 5/4 Friday Schedule
Loka runs a 5/4 schedule: every other Friday is off. That is 26 extra days off per year on top of regular vacation. Roughly one long weekend every two weeks, permanently, forever. Work-life balance is not a perk here, it is the calendar.

Great question! Loka has a fantastic time-off policy that really prioritizes work-life balance:

**The 5/4 Friday Schedule**: Loka operates on a unique schedule where every other Friday is off. This gives you **26 extra days off per year** on top of regular vacation time. That means you get roughly one long weekend every two weeks, permanently!

At Loka, work-life balance isn't just a perk—it's built right into the calendar. Pretty cool, right? 😊


### 🎯 Ask your own

In [10]:
result = graph.invoke({
    "messages": [
        SystemMessage(SYSTEM_PROMPT),
        HumanMessage("What kind of projects does Loka's engineering team typically work on?"),
    ]
})
print(result["messages"][-1].content)

Great question! Based on what I found, Loka's engineering team works on **cutting-edge client projects** that leverage the newest AI tools and frameworks. This includes:

- **Generative AI** projects
- **Agent frameworks**
- **Modern cloud architectures**

The exciting part? You're not stuck maintaining legacy code—you're shipping with the latest tools that are being showcased at major tech conferences. It's the kind of work that keeps you on the cutting edge of technology!

Plus, as part of Loka's fully remote, global team, you'd be collaborating with talented engineers from all over the world. Pretty cool environment to grow your skills! 🚀


### 🚀 Going further

If you finished early or want to explore more, try these ideas:

- **Inspect the run.** Print `result["messages"]` in full to see how LangGraph
  represents the conversation as a list of typed messages (system/human/ai/tool).
- **Use a prebuilt LangChain tool** instead of a custom one, e.g. from
  `langchain_community.tools`. Add it to both the `ToolNode` and the
  `bind_tools([...])` list.
- **Change the persona** in `SYSTEM_PROMPT` (formal? pirate?) and re-run.
- **Ask something NOT in the knowledge base** and see if the agent admits it doesn't know.
- **Visualize the graph** with `graph.get_graph().print_ascii()`.

## Exercise 2: Multi-turn Agent

Two upgrades to the agent from Exercise 1:

1. **Memory** -- a **checkpointer** persists the conversation for you, keyed by
   a `thread_id`. Unlike Strands (where the `Agent` object just remembers for
   free), here memory is an explicit piece you attach when you compile the graph.
2. **Tool selection** -- two more tools. The router (`tools_condition`) is the
   same one from Exercise 1; what changes is that the model now has more tools
   to *choose* between.

The graph shape (`agent` -> router -> `tools` -> loop back) is identical to
Exercise 1, so it's given below, already wired. Your turns: add the two new
tools, and turn on memory.

> 💡 Use `graph.get_state(config).values` to inspect what a checkpointer has stored.

**📚 Hints**
- [Persistence / memory concepts](https://langchain-ai.github.io/langgraph/concepts/persistence/)

### ✅ Given: the updated system prompt

In [11]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import MessagesState

SYSTEM_PROMPT = """You are the Loka Research Agent, a friendly assistant that \
answers questions about Loka (the company).

Pick the right tool:
- list_knowledge_base_topics: overview / "what can you tell me about" questions.
- search_knowledge_base: internal facts about working at Loka (benefits, time \
off, remote culture, awards, learning).
- search_loka_website: public/marketing info (services and solutions Loka \
offers, job openings, how it positions itself).

It's a conversation: use earlier turns to resolve follow-ups ("that", "there").
If neither source has the answer, say so. Be concise, warm, and proud of Loka.
"""

### ✏️ Your turn -- TODO 1 & 2: two more tools

`search_knowledge_base` from Exercise 1 is still in scope -- reuse it below.

In [12]:
@tool
def list_knowledge_base_topics() -> str:
    """List the topics available in Loka's knowledge base. Use for overview or
    "what can you tell me about" questions rather than specific ones."""
    return list_topics()


@tool
def search_loka_website(query: str) -> str:
    """Search Loka's public website (loka.com) for what the company presents
    publicly: the services and solutions it offers, open roles, and
    positioning.

    Args:
        query: A short natural-language description of what to look for.
    """
    return search_website(query)

### ✏️ Your turn -- TODO 3: bind all three tools

In [13]:
ALL_TOOLS = [search_knowledge_base, list_knowledge_base_topics, search_loka_website]

model_with_tools = model.bind_tools(ALL_TOOLS)

### ✅ Given: the graph -- same shape as Exercise 1

In [14]:
def agent_node(state: MessagesState) -> dict:
    return {"messages": [model_with_tools.invoke(state["messages"])]}


builder = StateGraph(MessagesState)
builder.add_node("agent", agent_node)
builder.add_node("tools", ToolNode(ALL_TOOLS))

builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", tools_condition, {"tools": "tools", END: END})
builder.add_edge("tools", "agent")

### ✏️ Your turn -- TODO 4: turn on memory

Compiling with a **checkpointer** is what turns this into memory: it persists
the state (the message list) per `thread_id`, so each `invoke()` call only
needs to carry the new turn -- the graph looks up the rest.

In [15]:
graph = builder.compile(checkpointer=InMemorySaver())

### ✅ Given: run a multi-turn conversation

Run the following three questions in order, in the same `thread_id`. Notice
how different questions use different tools, and that the third question
works only because the agent remembers the second.

In [16]:
config = {"configurable": {"thread_id": "workshop-demo"}}

response = graph.invoke(
    {"messages": [SystemMessage(SYSTEM_PROMPT), HumanMessage("What can you tell me about Loka?")]},
    config,
)
print(response["messages"][-1].content)

Great question! Here's what I can tell you about **Loka**:

## What Loka Does
Loka is a tech services company that helps clients turn their ideas into reality faster. We specialize in:
- **GenAI & ML** – Generative AI and machine learning solutions
- **Data & Analytics** – Data-driven insights and solutions
- **DevOps & Migrations** – Cloud infrastructure and modernization
- **Design & App Development** – User experience and software development
- **Healthcare & Life Sciences** – Specialized solutions for the healthcare sector
- **AWS Solutions** – We're an AWS Innovation Partner of the Year

## What Makes Loka Special
- **Fully Remote, Work From Anywhere** – We're a distributed global team with flexibility
- **Multicultural, Global Team** – Diverse talent from around the world
- **Cutting-Edge Projects** – We work on innovative, impactful client work
- **Strong Culture** – Known for innovation, learning, and development opportunities
- **Great Work-Life Balance** – Including a unique 

In [17]:
response = graph.invoke(
    {"messages": [HumanMessage("What has AWS recognized Loka for?")]},
    config,
)
print(response["messages"][-1].content)  # expect: the knowledge base tool

Excellent question! AWS recognized Loka as **AWS Innovation Partner of the Year**.

This is a big deal—it means that when Amazon Web Services wants to highlight someone doing genuinely novel and cutting-edge work on their cloud platform, they point to Loka. 

What this recognition reflects:
- Loka is building innovative solutions using cutting-edge AWS services like **Bedrock, SageMaker, and Lambda**
- We have **direct partner access** with AWS, which benefits our clients
- We're doing genuinely novel things that stand out in the AWS partner ecosystem

It's a testament to Loka's culture of innovation and our commitment to staying at the forefront of cloud technology. We even have a dedicated R&D department and actively encourage internal initiatives—if you have an idea, there's a path for it to become reality.

Pretty cool recognition! 🚀


In [18]:
response = graph.invoke(
    {"messages": [HumanMessage("How would that show up in the projects engineers work on?")]},  # requires memory
    config,
)
print(response["messages"][-1].content)

Based on what I know, that AWS Innovation Partner recognition shows up in Loka's project work in a few concrete ways:

1. **Cutting-Edge AWS Services** – Engineers work with the latest AWS technologies like **Bedrock** (for generative AI), **SageMaker** (for machine learning), and **Lambda** (for serverless computing). These aren't just standard implementations—they're the frontier tools that AWS itself highlights.

2. **Direct Partner Access** – Because of our AWS partnership status, engineers get direct access to AWS resources and support, which means better collaboration and potentially early access to new features or capabilities.

3. **Innovative Client Projects** – The knowledge base mentions that Loka works on "cutting-edge client projects," and this AWS recognition is a big part of why. Engineers are solving genuinely novel problems, not just standard implementations.

4. **Culture of Innovation** – Beyond just the tools, there's an organizational commitment to experimentation 

In [19]:
history = graph.get_state(config).values["messages"]
print(f"Memory holds {len(history)} messages — the checkpointer kept them for us.\n")

Memory holds 12 messages — the checkpointer kept them for us.



### 🎯 Ask your own

In [20]:
response = graph.invoke(
    {"messages": [HumanMessage("Is Loka hiring right now, and what's the interview process like?")]},
    config,
)
print(response["messages"][-1].content)
print("Memory now holds", len(graph.get_state(config).values["messages"]), "messages")

Good news—Loka definitely has a **Careers page** at [loka.com/careers](https://www.loka.com/careers), so they are actively hiring!

However, I don't have specific details in my knowledge base about the interview process itself. For the most current information on:
- Open positions
- Specific roles being hired for
- Interview process and what to expect

I'd recommend visiting **[loka.com/careers](https://www.loka.com/careers)** directly. That's where you'll find all the active job openings and can likely get details about the hiring process.

What I *can* tell you is that Loka values:
- **Cutting-edge technical skills** (especially around AWS, AI/ML, DevOps, etc.)
- **Innovation and creativity** – they encourage people who want to build novel solutions
- **Global, multicultural perspectives** – they have a diverse team across the world
- **Learning mindset** – they invest in learning and development

If you're interested in applying, the careers page is your best starting point! Feel fr

### 🚀 Going further (optional)

If you finished early or want to explore more, try these ideas:

- **Try an ambiguous follow-up** and see which tool it picks.
- **Peek at memory.** Print `graph.get_state(config).values["messages"]` to see
  exactly how LangGraph stores the conversation.
- **Start a second conversation** with a different `thread_id` and confirm it's
  isolated from the first.
- **Trim the history yourself.** LangGraph doesn't auto-manage a sliding window
  like Strands' conversation manager -- you'd slice `state["messages"]` in the
  node, or reach for `trim_messages` from `langchain_core.messages`.
- **Modify the system prompt** to change how follow-ups are handled.

## Exercise 3: Human-in-the-loop

Strands has no built-in equivalent for this: pausing execution before a side
effect, then resuming later once a human has decided. LangGraph builds it in
via `interrupt()` + a checkpointer.

The graph shape is *still* the same as Exercises 1-2 (`agent` -> router ->
`tools` -> loop) -- `interrupt()` lives inside one tool's body, and the plain
`ToolNode` you already know pauses there for you. No new node type needed.

Scenario: the agent can still research Loka (same two read-only tools), and
now can also draft a summary email. Sending an email is irreversible enough to
want a human to approve it first.

**📚 Hints**
- [Human-in-the-loop / `interrupt()` concepts](https://langchain-ai.github.io/langgraph/concepts/human_in_the_loop/)

### ✅ Given: the read-only tools, reused from before, and the updated prompt

> 💡 Why does the prompt say "do NOT draft the email as chat text"? Without
> that line, the model tends to write the email as a normal chat answer and
> ask "does this look good?" instead of calling the tool -- which means the
> interrupt never fires, since it lives inside the tool. Worth trying with and
> without that line to see the difference.

In [21]:
from langgraph.types import Command, interrupt

APPROVE_WORDS = {"y", "yes", "approve", "approved"}

SYSTEM_PROMPT = """You are the Loka Research Agent, a thorough assistant that \
answers questions about Loka (the company).

Gather evidence with search_knowledge_base and search_loka_website before \
answering. If asked to email a summary, call send_research_summary_email \
directly with a concise subject and body grounded in what you found -- do NOT \
draft the email as chat text and ask the user to confirm it first. A human \
reviewer approves or rejects the call before it actually sends, so calling the \
tool *is* how you propose the email; write it as if it will be sent as-is.
"""

### ✏️ Your turn -- TODO 1: the tool that pauses

Write `send_research_summary_email`. It should call `interrupt(...)` with the
drafted email so a human can review it, then check the returned decision
before "sending".

In [22]:
@tool
def send_research_summary_email(to: str, subject: str, body: str) -> str:
    """Send an email with a research summary. This has a real side effect (an
    email leaves the building), so it pauses for human approval before
    actually sending -- gather evidence with the other tools before drafting
    one.

    Args:
        to: Recipient email address.
        subject: Email subject line.
        body: The email body -- a concise summary grounded in what you found.
    """
    decision = interrupt({"to": to, "subject": subject, "body": body})
    if str(decision).strip().lower() not in APPROVE_WORDS:
        return "The human reviewer did not approve this email. It was not sent."

    print(f"\n*** EMAIL SENT ***\nTo: {to}\nSubject: {subject}\n\n{body}\n")
    return f"Email to {to} sent successfully."

### ✏️ Your turn -- TODO 2: add the tool and turn memory back on

The graph wiring below is the same shape you've now built twice -- given as-is.
Add the new tool to the list, and compile with a checkpointer again:
`interrupt()` needs one to pause and resume the graph across separate
`invoke()` calls.

In [23]:
ALL_TOOLS = [search_knowledge_base, search_loka_website, send_research_summary_email]
model_with_tools = model.bind_tools(ALL_TOOLS)


def agent_node(state: MessagesState) -> dict:
    return {"messages": [model_with_tools.invoke(state["messages"])]}


builder = StateGraph(MessagesState)
builder.add_node("agent", agent_node)
builder.add_node("tools", ToolNode(ALL_TOOLS))
builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", tools_condition, {"tools": "tools", END: END})
builder.add_edge("tools", "agent")

graph = builder.compile(checkpointer=InMemorySaver())

### ✅ Given: the approve/reject loop

`interrupt()` shows up in the result under the `"__interrupt__"` key; resuming
is just another `invoke()` call with `Command(resume=...)`.

In [24]:
def run_until_done(initial_input, config):
    result = graph.invoke(initial_input, config)

    while "__interrupt__" in result:
        request = result["__interrupt__"][0].value
        print("\n--- Approval needed ---")
        print(f"Send email to {request['to']}?")
        print(f"Subject: {request['subject']}")
        print(f"Body: {request['body']}")
        answer = input("Approve? [y/N] > ").strip()
        result = graph.invoke(Command(resume=answer), config)

    print()
    print(result["messages"][-1].content)

### ✅ Given: run it

In [25]:
config = {"configurable": {"thread_id": "workshop-demo-3"}}

question = (
    "Look up Loka's time-off policy and email a short summary to "
    "hr@loka.com so they can review it."
)

run_until_done(
    {"messages": [SystemMessage(SYSTEM_PROMPT), HumanMessage(question)]},
    config,
)


--- Approval needed ---
Send email to hr@loka.com?
Subject: Loka Time-Off Policy Summary
Body: Hi HR Team,

Here's a summary of Loka's time-off policy:

Loka operates on a 5/4 Friday schedule, where every other Friday is off. This provides employees with 26 additional days off per year on top of regular vacation time. This results in approximately one long weekend every two weeks on a permanent basis.

The policy reflects Loka's commitment to work-life balance as a core part of the company calendar, rather than as an optional perk.

Best regards,
Loka Research Agent

*** EMAIL SENT ***
To: hr@loka.com
Subject: Loka Time-Off Policy Summary

Hi HR Team,

Here's a summary of Loka's time-off policy:

Loka operates on a 5/4 Friday schedule, where every other Friday is off. This provides employees with 26 additional days off per year on top of regular vacation time. This results in approximately one long weekend every two weeks on a permanent basis.

The policy reflects Loka's commitment to

### 🎯 Ask your own

In [26]:
question = (
    "What does Loka offer in terms of learning and development, and can you "
    "email a short summary to hr@loka.com?"
)

run_until_done(
    {"messages": [SystemMessage(SYSTEM_PROMPT), HumanMessage(question)]},
    {"configurable": {"thread_id": "workshop-demo-3-mine"}},
)


--- Approval needed ---
Send email to hr@loka.com?
Subject: Loka's Learning and Development Offerings
Body: Hi,

Here's a summary of Loka's learning and development offerings:

**Learning & Development Commitment:**
Loka has made becoming one of the world's best learning organizations a headline goal for 2026. The company allocates real budget and real time for employee growth, including courses and certifications. Learning is treated as an integral part of the job, not an optional extra.

**Culture of Continuous Growth:**
Beyond formal programs, Loka fosters a culture of innovation and internal initiatives. Employees are encouraged to experiment and build new things, with clear pathways for ideas to be developed and implemented.

**Work-Life Balance Support:**
The company operates a 5/4 Friday schedule, providing every other Friday off—equivalent to 26 extra days off annually on top of regular vacation. This reflects Loka's commitment to work-life balance as a core value.

Best regar

### 🚀 Going further (optional)

- **Reject the email** and see the exact message the model gets back, then ask
  it to revise and try again.
- **Inspect the pause.** Right after the interrupt fires, before resuming,
  print `graph.get_state(config)` and look at `.next` and `.tasks` to see
  exactly where the graph is paused.
- **Add a second interrupt-guarded tool** (e.g. "post an announcement") and
  see it work with the same `run_until_done` loop -- no new plumbing needed.
- **Compare to Strands.** What would you have to build yourself to make the
  Strands agent pause for approval the same way?

## Conclusion

You built the same Loka Research Agent as the Strands notebook, but wired
explicitly: a state, a node that calls the model, a router, a node that runs
tools, and -- from Exercise 2 on -- a checkpointer for memory. The graph shape
barely changed across exercises: what grew was the tool list and, in Exercise
3, one new kind of pause.

#### Questions for further reflection:

- How much of what you wrote each exercise was really new, versus the same few
  moves repeated (`bind_tools`, `add_node`/`add_edge`, `compile(checkpointer=...)`)?
- LangGraph made the human-in-the-loop pause almost free once the graph
  existed. Would that have been easy to bolt onto the Strands agent? What would
  you need to build yourself?
- Compare with the Strands notebook: where did explicit control (LangGraph)
  feel worth the extra code, and where did it feel like ceremony for something
  the model could have just decided on its own?